---
title: "Routing, Validation, and Recovery"
draft: true
categories: [agents, workflows, langgraph, ai-engineering, search, retrieval, reliability]
---

Invalid output is a workflow state, not an exception to hide. A transient search failure, stale index, incomplete branch, disallowed test, and ambiguous result have different owners and therefore different routes.


## Failure classes become trajectories

The fixture injects failures at the boundaries where a production investigation can become misleading. Retry is reserved for a transient branch error; stale evidence, incomplete fan-in, and policy restrictions terminate explicitly.


In [1]:
import sys
from pathlib import Path

from change_planner.schemas import FaultPlan
from change_planner.verification import run_targeted_test
from change_planner.workflow import run_fixture


scenarios = {
    "happy path": FaultPlan(),
    "timeout then success": FaultPlan(transient_failures=1),
    "stale index": FaultPlan(stale_index=True),
    "missing tests branch": FaultPlan(missing_task_id="tests"),
    "tests disallowed": FaultPlan(disallow_tests=True),
}
results = {name: run_fixture("dry-run-01", faults=faults) for name, faults in scenarios.items()}
for name, state in results.items():
    print(name, "->", state.get("status"), "/", state.get("terminal_reason"))
assert results["happy path"]["status"] == "complete"
assert results["timeout then success"]["status"] == "complete"
assert results["stale index"]["terminal_reason"] == "index is stale for target revision"
assert results["missing tests branch"]["status"] == "failed"

verification = run_targeted_test(
    Path(".tmp"),
    [sys.executable, "-c", "print('targeted check')"],
    authorized=True,
)
blocked = run_targeted_test(Path(".tmp"), ["sh", "-c", "echo unsafe"], authorized=True)
print({"verification": verification.model_dump(), "blocked": blocked.reason})
assert verification.status == "passed"
assert verification.output_fingerprint.startswith("sha256:")
assert blocked.status == "blocked"


happy path -> complete / completion contract satisfied
timeout then success -> complete / completion contract satisfied
stale index -> failed / index is stale for target revision
missing tests branch -> failed / incomplete investigation branches: tests
tests disallowed -> failed / targeted test execution disallowed by policy
{'verification': {'command': ['/Users/particle1331/code/latest/watchtower/.venv/bin/python3', '-c', "print('targeted check')"], 'cwd': '/Users/particle1331/code/latest/watchtower/.tmp', 'status': 'passed', 'returncode': 0, 'output_fingerprint': 'sha256:01f2cf04183052cb35a7920b28f612fd7a250db8ea77fd32d726da85a532580f', 'stdout': 'targeted check\n', 'stderr': '', 'duration_ms': 34, 'reason': 'command completed successfully'}, 'blocked': 'program not allowed: sh'}


The timeout is retryable because repeating retrieval can change the outcome. The stale-index and missing-branch cases are not repaired by blind repetition: they require a rebuild, a replan, or an explicit incomplete result. The run record makes each decision inspectable.

## Typed recovery beats blind retry


In [2]:
comparison = [
    {"strategy": "blind retry", "stale_index": "hidden", "missing_branch": "hidden", "duplicate_effects": 2},
    {"strategy": "typed routes", "stale_index": "terminal", "missing_branch": "terminal", "duplicate_effects": 0},
]
for row in comparison:
    print(row)
assert results["timeout then success"]["run_metrics"]["attempts"]["behavior"] == 2
assert len([effect for effect in results["timeout then success"]["run_metrics"]["effects"] if effect.startswith("search:")]) == 4


{'strategy': 'blind retry', 'stale_index': 'hidden', 'missing_branch': 'hidden', 'duplicate_effects': 2}
{'strategy': 'typed routes', 'stale_index': 'terminal', 'missing_branch': 'terminal', 'duplicate_effects': 0}


Explicit counters and terminal reasons are the primary loop bounds; the runtime recursion limit remains a final safety net. Chapter 05 adds the failure surface of parallel investigation by requiring the join to account for every scheduled branch.
